# Personal Finance & Spending Analysis
## 01 — Data Exploration

**Goal:** Explore personal income, household expenses, and savings-related data before cleaning and deeper analysis.

**Workflow:** Import libraries → load raw data → inspect structure → check data quality → summarize variables → explore distributions and relationships → record observations.

**Input:** `data/raw_data.csv`  
**Important:** This notebook does not modify or overwrite the raw CSV. It supports launching from either the project root or `notebooks/`.

## 1. Environment and Libraries

If VS Code reports that `ipykernel` is missing, run this in its terminal using the Python environment selected for the notebook:

```bash
python -m pip install --upgrade pip
python -m pip install ipykernel pandas numpy matplotlib seaborn
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
print("Libraries imported successfully.")

## 2. Locate and Load Raw Dataset

In [ ]:
from pathlib import Path
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "data").exists() else HERE.parent
print("Project root:", PROJECT_ROOT.resolve())

DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}. Place raw_data.csv in data/.")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]:,} columns from {DATA_PATH.resolve()}")

## 3. Initial Data Inspection

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
if len(df):
    display(df.sample(n=min(5, len(df)), random_state=42))
else:
    print("Dataset contains no records.")

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
for i, column in enumerate(df.columns, 1):
    print(f"{i:02d}. {column}")

In [ ]:
df.info()

## 4. Data Quality Checks

### 4.1 Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})
missing_summary = missing_summary[missing_summary.missing_count > 0].sort_values(
    "missing_percentage", ascending=False
)
if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary.style.format({"missing_percentage": "{:.2f}%"}))

### 4.2 Duplicate Records

In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_pct = duplicate_count / len(df) * 100 if len(df) else 0
print(f"Duplicate rows: {duplicate_count:,} ({duplicate_pct:.2f}%)")

### 4.3 Data Types and Expected Numeric Fields

In [ ]:
display(pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values
}))

In [ ]:
expected_numeric = [
    "Income", "Age", "Dependents", "Rent", "Loan_Repayment", "Insurance",
    "Groceries", "Transport", "Eating_Out", "Entertainment", "Utilities",
    "Healthcare", "Education", "Miscellaneous", "Desired_Savings_Percentage",
    "Desired_Savings", "Disposable_Income", "Potential_Savings_Groceries",
    "Potential_Savings_Transport", "Potential_Savings_Eating_Out",
    "Potential_Savings_Entertainment", "Potential_Savings_Utilities",
    "Potential_Savings_Healthcare", "Potential_Savings_Education",
    "Potential_Savings_Miscellaneous"
]
display(pd.DataFrame([
    {"column": c, "present": c in df.columns,
     "dtype": str(df[c].dtype) if c in df.columns else "Not present",
     "is_numeric": pd.api.types.is_numeric_dtype(df[c]) if c in df.columns else False}
    for c in expected_numeric
]))

## 5. Statistical Summary

In [ ]:
display(df.describe().T)

In [ ]:
display(df.describe(include='all').T)

## 6. Categorical Variable Exploration

In [ ]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()
print("Categorical columns:", categorical_columns or "None detected")
for column in categorical_columns:
    print(f"\\n--- {column} ---")
    display(df[column].value_counts(dropna=False).rename("count").to_frame())

## 7. Numerical Variable Distributions

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
print("Numerical columns detected:", len(numerical_columns))
if numerical_columns:
    df[numerical_columns].hist(
        bins=25,
        figsize=(16, max(6, 3 * int(np.ceil(len(numerical_columns) / 3)))),
        edgecolor="black"
    )
    plt.suptitle("Distributions of Numerical Variables", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns available.")

## 8. Potential Outlier Review
Boxplots visually flag unusual values. Outliers are not automatically errors; review them in business context.

In [ ]:
for start in range(0, len(numerical_columns), 8):
    cols = numerical_columns[start:start + 8]
    plot_data = df[cols].melt(var_name="variable", value_name="value")
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=plot_data, x="variable", y="value")
    plt.title("Boxplots — " + ", ".join(cols))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Correlation Analysis
Correlation measures linear association; it does not prove causation.

In [ ]:
if len(numerical_columns) >= 2:
    corr = df[numerical_columns].corr()
    plt.figure(figsize=(16, 12))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.3)
    plt.title("Correlation Heatmap of Numerical Variables")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are required.")

## 10. Finance-Focused Checks

In [ ]:
finance_columns = [
    "Income", "Disposable_Income", "Desired_Savings", "Desired_Savings_Percentage",
    "Rent", "Groceries", "Transport", "Eating_Out", "Entertainment", "Utilities",
    "Healthcare", "Education", "Miscellaneous", "Loan_Repayment", "Insurance"
]
available = [c for c in finance_columns if c in df.columns]
missing = [c for c in finance_columns if c not in df.columns]
print("Available:", available)
print("Not present:", missing)
if available:
    display(df[available].describe().T)

In [ ]:
if {"Occupation", "Income"}.issubset(df.columns):
    display(df.groupby("Occupation", dropna=False)["Income"]
            .agg(["count", "mean", "median", "min", "max"])
            .sort_values("mean", ascending=False))
else:
    print("Occupation and/or Income not found; skipped.")

In [ ]:
if {"City_Tier", "Income"}.issubset(df.columns):
    display(df.groupby("City_Tier", dropna=False)["Income"]
            .agg(["count", "mean", "median"])
            .sort_values("mean", ascending=False))
else:
    print("City_Tier and/or Income not found; skipped.")

## 11. Initial Observations

Complete these notes after running the notebook. Use actual results, not assumptions.

- Dataset size:
- Data quality (missing values and duplicates):
- Income distribution:
- Expense patterns:
- Unusual values or data types:
- Questions to investigate during cleaning:

**Next:** Run `02_data_cleaning.ipynb`. The original raw file remains unchanged.